In [ ]:
# %% [markdown]

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MattArtzAnthro/wikidata-tools/blob/main/notebooks/Wikidata_Anthropology_Main_Subject_Counter.ipynb)
# # Wikidata Anthropology Main Subject Counter
#
# Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)
#
# ---
#
# ## What This Notebook Does
#
# Counts how many scholarly articles in specified journals have "anthropology" (Q23404) as a main subject (P921) in Wikidata. Useful for assessing subject tagging completeness and prioritizing metadata enrichment.
#
# Includes 36 anthropology journal QIDs by default. Results include summary statistics, bar chart visualization, and identification of journals with no tagged articles.
#
# **Endpoint note**: Queries the scholarly endpoint (`query-scholarly.wikidata.org`) required since the May 2025 graph split.
#
# ## Workflow
#
# 1. **Configure**: Set journal QIDs and endpoint parameters
# 2. **Query**: Single SPARQL query counts articles with P921=Q23404 per journal
# 3. **Review**: Summary stats, results table, and bar chart
# 4. **Identify gaps**: Journals with zero tagged articles listed separately
# 5. **Export**: Download timestamped CSV
#
# ## Citation
#
# If you use this notebook, please cite:
#
# > Artz, Matt. (2026). MattArtzAnthro/wikidata-tools. Zenodo. https://doi.org/10.5281/zenodo.18912858
#
# ## License
#
# [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

# %% [markdown]
# ## Setup

# %%
# Install required packages
!pip install requests pandas matplotlib -q

import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display, HTML
import time

print("Setup complete.")

# %% [markdown]
# ## Configuration
#
# **Important**: Since May 2025, Wikidata underwent a graph split. Scholarly articles are now **only** available on the scholarly endpoint.

# %%
# Wikidata Scholarly Query Service endpoint
SCHOLARLY_ENDPOINT = "https://query-scholarly.wikidata.org/sparql"

# User-Agent for API requests (required by Wikidata)
USER_AGENT = "WikidataAnthropologyMainSubjectCounter/1.0 (https://www.mattartz.me/)"

# Request timeout in seconds
TIMEOUT = 60

# Journal QIDs to query
JOURNAL_QIDS = [
    "Q4579783",
    "Q15762644",
    "Q27724868",
    "Q15754091",
    "Q15755065",
    "Q53952408",
    "Q4773909",
    "Q15766859",
    "Q15759624",
    "Q96698430",
    "Q27718969",
    "Q15752788",
    "Q15716560",
    "Q63871858",
    "Q137524520",
    "Q73541368",
    "Q96697459",
    "Q27723450",
    "Q15757300",
    "Q96701409",
    "Q5531693",
    "Q73541374",
    "Q15752869",
    "Q15760155",
    "Q15758525",
    "Q6806259",
    "Q15752491",
    "Q96720358",
    "Q63871800",
    "Q96728685",
    "Q96731837",
    "Q137524523",
    "Q96734245",
    "Q96735356",
    "Q27722507",
    "Q96733008",
]

print(f"Configured {len(JOURNAL_QIDS)} journals to query")

# %% [markdown]
# ## SPARQL Query Construction

# %%
def build_sparql_query(journal_qids):
    """
    Build SPARQL query to count articles with main subject anthropology (Q23404)
    for the specified journals.
    """
    qid_values = "\n    ".join([f"wd:{qid}" for qid in journal_qids])

    query = f"""
SELECT ?journal ?journalLabel (COUNT(?article) AS ?articleCount) WHERE {{
  VALUES ?journal {{
    {qid_values}
  }}
  ?article wdt:P1433 ?journal ;
           wdt:P921 wd:Q23404 .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
GROUP BY ?journal ?journalLabel
ORDER BY DESC(?articleCount)
"""
    return query

sparql_query = build_sparql_query(JOURNAL_QIDS)
print("SPARQL query constructed")
print("\n--- Query Preview (first 500 chars) ---")
print(sparql_query[:500] + "...")

# %% [markdown]
# ## Execute Query

# %%
def execute_sparql_query(endpoint, query, user_agent, timeout):
    """Execute a SPARQL query and return results as a DataFrame."""
    headers = {
        "User-Agent": user_agent,
        "Accept": "application/sparql-results+json"
    }

    params = {
        "query": query,
        "format": "json"
    }

    print(f"Querying: {endpoint}")

    try:
        response = requests.get(
            endpoint,
            params=params,
            headers=headers,
            timeout=timeout
        )
        response.raise_for_status()

        data = response.json()
        results = data.get("results", {}).get("bindings", [])

        if not results:
            print("Query returned no results.")
            return pd.DataFrame()

        rows = []
        for result in results:
            row = {
                "journal_qid": result.get("journal", {}).get("value", "").replace("http://www.wikidata.org/entity/", ""),
                "journal_label": result.get("journalLabel", {}).get("value", ""),
                "article_count": int(result.get("articleCount", {}).get("value", 0))
            }
            rows.append(row)

        df = pd.DataFrame(rows)
        print(f"Query successful: {len(df)} journals returned with tagged articles")
        return df

    except requests.exceptions.Timeout:
        print(f"Query timed out after {timeout} seconds.")
        return pd.DataFrame()
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return pd.DataFrame()

# Execute the query
results_df = execute_sparql_query(SCHOLARLY_ENDPOINT, sparql_query, USER_AGENT, TIMEOUT)

# %% [markdown]
# ## Results

# %%
if not results_df.empty:
    total_articles = results_df["article_count"].sum()
    journals_with_articles = len(results_df)
    journals_without = len(JOURNAL_QIDS) - journals_with_articles

    print("=" * 50)
    print("SUMMARY")
    print("=" * 50)
    print(f"Total articles with main subject 'anthropology': {total_articles:,}")
    print(f"Journals with tagged articles: {journals_with_articles}")
    print(f"Journals without tagged articles: {journals_without}")
    if journals_with_articles > 0:
        print(f"Average per journal (with articles): {total_articles / journals_with_articles:.1f}")
    print("=" * 50)

    print("\n")
    display(HTML("<h3>Articles by Journal</h3>"))
    display(results_df.style.format({"article_count": "{:,}"}).hide(axis="index"))
else:
    print("No results to display.")

# %% [markdown]
# ## Visualization

# %%
if not results_df.empty and len(results_df) > 0:
    fig, ax = plt.subplots(figsize=(12, max(8, len(results_df) * 0.4)))

    plot_df = results_df.sort_values("article_count", ascending=True)

    bars = ax.barh(plot_df["journal_label"], plot_df["article_count"], color="#4a90d9")

    for bar, count in zip(bars, plot_df["article_count"]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{count:,}', va='center', fontsize=9)

    ax.set_xlabel("Number of Articles with Main Subject: Anthropology (Q23404)")
    ax.set_title("Scholarly Articles Tagged with 'Anthropology' by Journal\n(Wikidata Scholarly Query Service)")

    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")

# %% [markdown]
# ## Export Results

# %%
if not results_df.empty:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"anthropology_main_subject_counts_{timestamp}.csv"

    export_df = results_df.copy()
    export_df["query_date"] = datetime.now().isoformat()
    export_df["main_subject_qid"] = "Q23404"
    export_df["main_subject_label"] = "anthropology"

    export_df.to_csv(filename, index=False)
    print(f"Saved: {filename} ({len(export_df)} journals)")

    try:
        from google.colab import files
        files.download(filename)
    except ImportError:
        print("(File saved to working directory)")
else:
    print("No results to export.")

# %% [markdown]
# ## Journals Without Results
#
# The main query only returns journals with at least one tagged article. This identifies which journals have no articles with the anthropology main subject.

# %%
if not results_df.empty:
    returned_qids = set(results_df["journal_qid"].tolist())
    missing_qids = [qid for qid in JOURNAL_QIDS if qid not in returned_qids]

    if missing_qids:
        print(f"Journals with NO articles tagged 'anthropology' ({len(missing_qids)}):")
        print("-" * 50)
        for qid in missing_qids:
            print(f"  - {qid}: https://www.wikidata.org/wiki/{qid}")
    else:
        print("All queried journals have at least one article with anthropology main subject.")
else:
    print("No query results to compare against.")